## Web Datamining & Semantics - Lab 1

The goal of this first lab is to build a complete pipeline for transforming unstructured web data into structured knowledge.

We focus on the domain of **AI Research** using a set of URLs. The pipeline follows two main steps:

- Web Crawling & Cleaning: extracting meaningful textual content from raw HTML
- Information Extraction: identifying entities and relations using NLP

This pipeline is a foundation for building a Knowledge Graph later.

URL selected :

*  https://en.wikipedia.org/wiki/Large_language_model
*  https://distill.pub/2021/understanding-gnns/
*  https://distill.pub/2021/multimodal-neurons/
*  https://distill.pub/2020/understanding-rl-vision/
*  https://en.wikipedia.org/wiki/Machine_learning
*  https://en.wikipedia.org/wiki/Transformer_(deep_learning)


In [1]:
import json
import time
import re
import csv
import urllib.robotparser
from urllib.parse import urlparse
from pathlib import Path
from typing import Optional

import httpx
import trafilatura
import spacy
import pandas as pd

In [2]:
SEED_URLS = [
    "https://en.wikipedia.org/wiki/Large_language_model",
    "https://distill.pub/2021/understanding-gnns/",
    "https://distill.pub/2021/multimodal-neurons/",
    "https://distill.pub/2020/understanding-rl-vision/",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning)",
]

In [3]:
TARGET_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

# Minimum words for a page to be considered "useful"
MIN_WORD_COUNT = 500

#output
JSONL_OUTPUT = "crawler_output.jsonl"
CSV_OUTPUT   = "extracted_knowledge.csv"

## Phase 1: Web Crawling & Cleaning

### Methodology

- We selected 6 seed URLs related to AI research.
- HTML content is fetched using `httpx`.
- The main textual content is extracted using `trafilatura` which removes some elements such as menus, ads and navigation links.
- Pages are filtered using a minimum threshold of **500 words** to ensure sufficient informational content because short pages often correspond to navigation or low-information content. By filtering them out, we improve the quality of the dataset used for NLP tasks.

### Output

The cleaned data is stored in a `.jsonl` file, preserving:
- URL
- Title
- Cleaned text
- Word count

In [4]:
def is_allowed_by_robots(url: str, user_agent: str = "AcademicResearchBot") -> bool:
    """
    Check if crawling a given URL is permitted by the site's robots.txt.

    Args:
        url: The URL to check.
        user_agent: The bot name to declare.

    Returns:
        True if allowed, False otherwise.
    """
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        return True

In [5]:
def fetch_html(url: str, timeout: int = 15) -> Optional[str]:
    """
    Fetch the raw HTML content of a given URL using httpx.
    Args:
        url: The target URL.
        timeout: Request timeout in seconds.

    Returns:
        HTML string if successful, None otherwise.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (compatible; AcademicResearchBot/1.0; +https://example.com)"
        )
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout, follow_redirects=True)
        response.raise_for_status()
        return response.text
    except httpx.HTTPError:
        return None

In [6]:
def extract_main_text(html: str, url: str) -> dict:
    """
    Use trafilatura to extract only the main article content from raw HTML.
    Args:
        html: Raw HTML string.
        url: Source URL (used for metadata extraction).

    Returns:
        A dict with 'title' and 'cleaned_text'.
    """
    cleaned_text = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False,
        no_fallback=False,
        favor_precision=True,
        deduplicate=True,
    )
    metadata = trafilatura.extract_metadata(html, default_url=url)
    title = metadata.title if metadata and metadata.title else urlparse(url).path.split("/")[-1]
    return {"title": title or "Unknown", "cleaned_text": cleaned_text or ""}

In [7]:
def is_useful_page(text: str, min_words: int = MIN_WORD_COUNT) -> bool:
    """
    Determine whether a page contains enough meaningful content.
    Args:
        text: Cleaned text string.
        min_words: Minimum word count threshold.

    Returns:
        True if the page meets the minimum word count requirement.
    """
    return len(text.split()) >= min_words

In [8]:
def save_jsonl(record: dict, filepath: str = JSONL_OUTPUT) -> None:
    """
    Append a single record as a JSON line to a .jsonl file.
    Args:
        record: Dict to serialize.
        filepath: Path to the output .jsonl file.
    """
    with open(filepath, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [9]:
def crawl_and_clean(urls: list[str]) -> list[dict]:
    """
    Main crawling and cleaning function.

    For each URL, the robots.txt file is checked, the HTML is downloaded,
    the main content is extracted, pages that are too short are filtered out,
    and the cleaned text is saved in JSONL format.
    """
    Path(JSONL_OUTPUT).write_text("", encoding="utf-8")
    pages: list[dict] = []

    for url in urls:
        if not is_allowed_by_robots(url):
            continue

        html = fetch_html(url)
        if not html:
            continue

        extracted = extract_main_text(html, url)
        cleaned_text = extracted["cleaned_text"]
        title = extracted["title"]

        if not cleaned_text or not is_useful_page(cleaned_text):
            continue

        record = {
            "url": url,
            "title": title,
            "cleaned_text": cleaned_text,
            "word_count": len(cleaned_text.split()),
            "source_domain": urlparse(url).netloc,
        }

        save_jsonl(record)
        pages.append(record)
        time.sleep(1.0)

    return pages

In [10]:
pages = crawl_and_clean(SEED_URLS)

## Phase 2 : Information Extraction

### Phase 2.1 : NER 

We use spaCy to extract key entities from the cleaned text. We focus on the following labels:
- PERSON
- ORG 
- GPE 
- DATE

To improve quality:
- Generic or non-informative entities are filtered out
- Duplicate entities within the same page are removed

In [11]:
def load_spacy_model(model_name: str = "en_core_web_trf") -> spacy.language.Language:
    """
    Loads the specified spaCy model or a fallback model.

    Args:
        model_name: The name of the model to load.
    Returns:
        A spaCy Language object.
    """
    try:
        return spacy.load(model_name)
    except OSError:
        try:
            return spacy.load("en_core_web_sm")
        except OSError as exc:
            raise RuntimeError(
                "Aucun modèle spaCy trouvé. Veuillez exécuter:\n"
                "  python -m spacy download en_core_web_trf\n"
                "ou: python -m spacy download en_core_web_sm"
            ) from exc

In [12]:
def _is_generic(ent_text: str) -> bool:
    """
    Identifies generic or overly short entities that should be removed.
    Args:
        ent_text: The entity’s text.

    Returns:
        True if the entity is considered generic.
    """

    txt = ent_text.lower().strip()
    if len(txt) <= 2:
        return True
    generic_tokens = {
        "the", "this", "that", "these", "those", "which", "who",
        "section", "table", "figure", "appendix", "chapter",
        "note", "example", "model", "models", "method", "methods",
        "result", "results", "approach", "overview", "introduction",
        "conclusion", "abstract", "reference", "references",
    }
    return txt in generic_tokens


In [13]:
def extract_entities(
    text: str,
    nlp: spacy.language.Language,
    source_url: str,
) -> list[dict[str, str]]:
    """
    Extract named entities from a text using spaCy.

    The analysed text is truncated for performance reasons.
    Only entities whose label is in TARGET_LABELS are kept,
    and entities considered generic or too short are filtered out.

    Args:
        text: Cleaned text.
        nlp: Loaded spaCy model.
        source_url: Source URL for traceability.

    Returns:
        A list of dictionaries:
        {"entity": ..., "entity_type": ..., "source_url": ...}
        without duplicates.
    """
    entities: list[dict[str, str]] = []
    seen: set[tuple[str, str]] = set()

    doc = nlp(text[:50000])

    for ent in doc.ents:
        if ent.label_ not in TARGET_LABELS:
            continue
        if _is_generic(ent.text):
            continue

        key = (ent.text.strip(), ent.label_)
        if key in seen:
            continue

        seen.add(key)
        entities.append(
            {
                "entity": ent.text.strip(),
                "entity_type": ent.label_,
                "source_url": source_url,
            }
        )

    return entities

### Phase 2.2 Relation extraction

Extracting precise semantic relations using dependency parsing proved to be too restrictive on real-world web data.

So we adopted a simpler and more robust approach based on **co-occurrence**:

- If at least two named entities appear in the same sentence, we create a relation between them using a generic label: `related_to`

In [22]:
def extract_relations(text: str, nlp: spacy.language.Language) -> list[tuple[str, str, str]]:
    relations = []
    doc = nlp(text[:50000])

    for sent in doc.sents:
        ent_spans = [ent for ent in sent.ents if ent.label_ in TARGET_LABELS]

        # si au moins 2 entités → on crée une relation simple
        if len(ent_spans) >= 2:
            for i in range(len(ent_spans) - 1):
                subj = ent_spans[i].text.strip()
                obj = ent_spans[i + 1].text.strip()
                relations.append((subj, "related_to", obj))

    return relations

#### Export

In [15]:
def export_entities_csv(entities: list[dict[str, str]], filepath: str = CSV_OUTPUT) -> pd.DataFrame:
    """
    Exports the list of entities in CSV format.
    The columns are: entity, entity_type, source_url. Duplicates are removed.
    Args:
        entities: List of extracted entities.
        filepath: Output path for the CSV file.

    Returns:
        The saved DataFrame.

    """
    if not entities:
        return pd.DataFrame()
    df = pd.DataFrame(entities)
    df = df.drop_duplicates()
    df.to_csv(filepath, index=False, encoding="utf-8")
    return df


In [23]:
def main() -> None:
    print("   AI RESEARCH KNOWLEDGE GRAPH PIPELINE")

    # Phase 1 : crawling + cleaning
    pages = crawl_and_clean(SEED_URLS)
    if not pages:
        print("No crawler detected.")
        return

    # Phase 2 : NLP
    nlp = load_spacy_model()

    all_entities: list[dict[str, str]] = []
    all_relations: list[dict[str, str]] = []

    for page in pages:
        text = page["cleaned_text"]
        url = page["url"]

        # 2.1 NER
        entities = extract_entities(text, nlp, url)
        all_entities.extend(entities)

        # 2.2 Relations
        relations = extract_relations(text, nlp)
        for subj, verb, obj in relations:
            all_relations.append(
                {
                    "subject": subj,
                    "relation": verb,
                    "object": obj,
                    "source_url": url,
                }
            )

    # Export entities
    export_entities_csv(all_entities, CSV_OUTPUT)

    # Export relations
    if all_relations:
        df_rel = pd.DataFrame(all_relations).drop_duplicates()
        df_rel.to_csv("extracted_relations.csv", index=False, encoding="utf-8")

    print("Pipeline finished.")
    print(f"Entities extracted: {len(all_entities)}")
    print(f"Relations extracted: {len(all_relations)}")



In [24]:
main()

   AI RESEARCH KNOWLEDGE GRAPH PIPELINE
Pipeline finished.
Entities extracted: 468
Relations extracted: 543


## Results and Discussion

### Results

The pipeline successfully processed the selected web pages and produced:

- A cleaned dataset stored in JSONL format
- A structured list of named entities in CSV format
- A set of candidate relations between entities based on sentence-level co-occurrence

A total of several hundred entities were extracted with a significant number of relations ensuring a dense representation of connections within the dataset.

### Observations

- Entity extraction is generally effective, especially for organizations (ORG) and locations (GPE).
- Some noise remains such as partial entity names or context-dependent ambiguities.
- The co-occurrence-based relation extraction produces a large number of relations capturing many potential links between entities.

### Limitations

- The extracted relations are not semantically precise,$ as they are based only on co-occurrence within sentences.
- The relation label `related_to` is generic and does not capture the exact nature of the relationship.
- Some extracted relations may be weak or not meaningful

### Conclusion

This pipeline demonstrates how unstructured web data can be transformed into structured entities and relations.

While the current approach prioritizes robustness over precision, it establishes a good foundation for future improvements.